АНАЛИЗ КАЧЕСТВА МОДЕЛИ ВОССТАНОВЛЕНИЯ ВЫСОТ ЗДАНИЙ 

In [ ]:
import pandas as pd
import yaml
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import ShuffleSplit, cross_validate, KFold
from ml_model import restore_building_height_model, predict_with_trained_model

Загрузка данных и настройка 

In [ ]:
# Загрузка конфигурационного файла 
def find_config(filename="config.yaml", start_dir=None):
    if start_dir is None:
        start_dir = os.getcwd()
    
    current_dir = start_dir
    while True:
        config_path = os.path.join(current_dir, filename)
        if os.path.exists(config_path):
            return config_path
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir: 
            break
        current_dir = parent_dir
    
    raise FileNotFoundError(f"{filename} не найден")

config_path = find_config("config.yaml")

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)


In [ ]:
WD = config['wd']
OUTPUT_DIR = os.path.join(WD, config['output_dir'])
MODELS_DIR = os.path.join(WD, config['models_dir'])

output_cfg = config['output']

ml_train_cfg = config.get('ml_train', {})

use_auto_tuning = ml_train_cfg.get('use_auto_tuning', False)
best_params_file = ml_train_cfg.get('model_params', 'catboost_best_params.json')
best_params_path = os.path.join(MODELS_DIR, best_params_file)

default_params = {
    'iterations': ml_train_cfg.get('iterations', 1000),
    'learning_rate': ml_train_cfg.get('learning_rate', 0.03),
    'depth': ml_train_cfg.get('depth', 4),
    'l2_leaf_reg': ml_train_cfg.get('l2_leaf_reg', 10)
}

if use_auto_tuning:
    with open(best_params_path, 'r') as f:
        best_params = json.load(f)
    print(f"Загружены лучшие гиперпараметры из {best_params_path}:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
else:
    best_params = default_params
    print("Используются параметры по умолчанию:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")

In [ ]:
predictions_csv = os.path.join(OUTPUT_DIR, output_cfg['predictions_csv'])
model_path = os.path.join(MODELS_DIR, output_cfg['model_file'])

In [ ]:
df = pd.read_csv(predictions_csv)
print(f"Загружено записей: {len(df)}")
model = CatBoostRegressor()
model.load_model(model_path)

# Числовые признаки
num_cols = [
    'area',
    'isoquotient',
    'obox_ratio',
    'width_med',
    'obox_hw',
    'min_dem',
    "neighbors_100m"
]

# Категориальные признаки
cat_cols = ['majority_100m', 'class', 'subtype']

# Целевая переменная
target_col = 'calc_height'

feature_cols = num_cols + cat_cols
print(f"\nПризнаки: {feature_cols}")


clean = df[feature_cols + [target_col]].dropna()

min_height = 1.0
clean = clean[clean[target_col] >= min_height]
print(f"После удаления пропусков: {len(clean)} записей")
print(f"Исключено зданий с высотой < {min_height} м: {len(df) - len(clean)}")

X = clean[feature_cols]
Y = clean[target_col]

for col in cat_cols:
    X[col] = X[col].astype(str)

cat_indices = [X.columns.get_loc(col) for col in cat_cols]

КРОСС-ВАЛИДАЦИЯ (ДЛЯ АВТОМАТИЧЕСКОГО ПОДБОРА ГИПЕРПАРАМЕТРОВ)

In [ ]:
cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=22)

mape_scorer = make_scorer(
    lambda y_true, y_pred: np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    greater_is_better=False
)

cv_results = cross_validate(
    X=X,
    y=Y,
    estimator=CatBoostRegressor(
        cat_features=cat_indices,
        loss_function='RMSE',
        random_seed=22,
        verbose=False,
        iterations=best_params.get('iterations'),
        learning_rate=best_params.get('learning_rate'),
        depth=best_params.get('depth'),
        l2_leaf_reg=best_params.get('l2_leaf_reg'),
        early_stopping_rounds=50,
        eval_metric='RMSE'
    ),
    cv=cv,
    return_train_score=True,
    scoring={
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error',
        'mape': mape_scorer
    }
)

print("\nРезультаты кросс-валидации:")

print(f"  R²:     {cv_results['test_r2'].mean():.4f}")
print(f"  RMSE:   {-cv_results['test_rmse'].mean():.2f}")
print(f"  MAPE:   {-cv_results['test_mape'].mean():.2f}%")


print(f"  R² тест:   {cv_results['test_r2'].mean():.4f}  |  R² трен:   {cv_results['train_r2'].mean():.4f}")
print(f"  RMSE тест: {-cv_results['test_rmse'].mean():.2f} м  |  RMSE трен: {-cv_results['train_rmse'].mean():.2f} м")
print(f"  MAPE тест: {-cv_results['test_mape'].mean():.2f}%  |  MAPE трен: {-cv_results['train_mape'].mean():.2f}% ")

АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ (ДЛЯ АВТОМАТИЧЕСКОГО ПОДБОРА ГИПЕРПАРАМЕТРОВ)

In [ ]:
model = CatBoostRegressor(
        cat_features=cat_indices,
        loss_function='RMSE',
        random_seed=22,
        verbose=False,
        iterations=best_params.get('iterations'),
        learning_rate=best_params.get('learning_rate'),
        depth=best_params.get('depth'),
        l2_leaf_reg=best_params.get('l2_leaf_reg'),
        early_stopping_rounds=50,
        eval_metric='RMSE'
)

model.fit(X, Y)

importances = model.feature_importances_
features = X.columns
sorted_idx = np.argsort(importances)

plt.figure(figsize=(10, 7))
plt.barh(features[sorted_idx], importances[sorted_idx], color='turquoise')
plt.xlabel("CatBoost Feature Importance")
plt.title(f"Важность признаков")
plt.yticks(fontsize=16)
plt.xticks(fontsize=12)
plt.tight_layout()
plt.show()

for i in sorted_idx[::-1]:
    print(f"  {features[i]}: {importances[i]:.2f}")